# Cancer Classification Using Machine Learning
### Breast Cancer Wisconsin Diagnostic Dataset
**Master's Bioinformatics • 2-Hour Jupyter Notebook Lab**

**Driving question:** Can machine learning distinguish malignant from benign breast tumors from diagnostic measurements, and how should we evaluate and interpret the model?

> Learning goals: understand the ML workflow, explore biomedical data, prevent data leakage, compare classifiers, evaluate beyond accuracy, and interpret predictive features responsibly.


## Lab Roadmap

1. Machine Learning Concept Check  
2. Load the Breast Cancer Wisconsin Dataset  
3. Understand Features and Target  
4. Exploratory Data Analysis (EDA)  
5. Train/Test Split  
6. Preprocessing  
7. Baseline: Logistic Regression  
8. Decision Tree and Random Forest  
9. Model Evaluation  
10. Confusion Matrix and ROC Curve  
11. Feature Importance  
12. Model Defense  
13. Mini Challenge  
14. Reflection


## Step 1 — Machine Learning Concept Check
Before coding, discuss with a partner:

- What is the difference between **classification** and **regression**?
- Why do we separate training and test data?
- What is **overfitting**?
- Why can **accuracy** be insufficient?
- What is **data leakage**?
- Does feature importance prove biological causality?

### Your answer
Write 2–3 sentences below about the concept you think is most important for biomedical machine learning.


**Student response:**  
_Type your answer here._


In [ ]:
# Step 1: Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, classification_report
)

RANDOM_STATE = 42

## Step 2 — Load the Dataset

Load the **Breast Cancer Wisconsin Diagnostic Dataset** from the project file `dataset/cancer.csv`. It contains measurements computed from digitized images of fine needle aspirates of breast masses. The dataset is from the [UCI Machine Learning Repository](https://doi.org/10.24432/C5DW2B).

**Citation:** Wolberg, W., Street, N., & Mangasarian, O. (1995). *Breast Cancer Wisconsin (Diagnostic)* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5DW2B.

- **569 samples**
- **30 numerical predictor features**
- **2 target classes:** malignant and benign
- 0 missing values in the supplied CSV file

### Research task
Predict whether a tumor is **malignant** or **benign**.


In [ ]:
from pathlib import Path

# Resolve the project-relative path whether the notebook runs from the project root or notebooks/.
dataset_path = Path("dataset/cancer.csv")
if not dataset_path.is_file():
    dataset_path = Path("../dataset/cancer.csv")
if not dataset_path.is_file():
    raise FileNotFoundError("Could not find dataset/cancer.csv from the project root or notebooks/ directory.")

data = pd.read_csv(dataset_path)
print("Dataset shape:", data.shape)
display(data.head())

In [ ]:
# Since 'data' is a pandas DataFrame read from cancer.csv:
# Extract features (X) by dropping the target column, and the target (y) as the target column
X = data.drop(columns=['target']).copy()
y = data['target'].copy()


print("X shape:", X.shape)
print("Target names:", ["malignant", "benign"])

### TODO 1 — Understand the dataset

Answer before continuing:

1. What does one **row** represent?
2. What does one **feature column** represent?
3. What is the target variable?
4. Is this classification or regression?


**Your answers**

1. Row: ...
2. Feature: ...
3. Target: ...
4. Problem type: ...


## Step 3 — Inspect Features and Target
The features describe cell-nucleus characteristics such as radius, texture, perimeter, area, smoothness, compactness, concavity, symmetry and fractal dimension.


In [ ]:
print("Number of features:", X.shape[1])
print("\nFeature names:")
for i, feature in enumerate(X.columns, start=1):
    print(f"{i:02d}. {feature}")

print("\nClass distribution:")
print(y.value_counts().rename(index={0: "malignant", 1: "benign"}))


## Step 4 — Exploratory Data Analysis (EDA)

Data mining begins by understanding the data before fitting a model.


In [ ]:
# Class distribution
class_counts = y.value_counts().sort_index()
labels = ["Malignant", "Benign"]

plt.figure(figsize=(6,4))
plt.bar(labels, class_counts.values)
plt.title("Class Distribution")
plt.ylabel("Number of Samples")
plt.show()


In [ ]:
# Descriptive statistics for selected features
selected_features = [
    "mean radius", "mean texture", "mean perimeter",
    "mean area", "mean concavity"
]
display(X[selected_features].describe().T)


In [ ]:
# Correlation matrix for selected features
corr = X[selected_features].corr()

plt.figure(figsize=(7,5))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(selected_features)), selected_features, rotation=60, ha="right")
plt.yticks(range(len(selected_features)), selected_features)
plt.colorbar(label="Correlation")
plt.title("Selected Feature Correlations")
plt.tight_layout()
plt.show()


### TODO 2 — EDA Interpretation

1. Are the classes perfectly balanced?
2. Which selected features appear strongly correlated?
3. Why might highly correlated predictors matter when interpreting a model?

**Your observations:**  
_Type your answer here._


## Step 5 — Train/Test Split

The test set must represent **unseen data**. We use stratification to preserve approximately the same class proportions in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


### TODO 3 — Experiment with the split

Change `test_size=0.20` to `0.30`, rerun the split, and observe the number of training/test samples.

**Question:** What is the trade-off when we allocate more data to the test set?

After experimenting, restore `test_size=0.20` before continuing.


## Step 6 — Preprocessing and Data Leakage

Some algorithms are sensitive to feature scale. We will standardize features for Logistic Regression.

**Important:** preprocessing must be learned from training data only. A `Pipeline` helps prevent leakage.


In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

logistic_model


### Think before running

Why is this safer?

`Training data → fit scaler → transform → train model`

instead of:

`Full dataset → fit scaler → split into train/test`


## Step 7 — Baseline Model: Logistic Regression
A baseline gives us a reference point before trying more complex models.


In [ ]:
logistic_model.fit(X_train, y_train)

lr_pred = logistic_model.predict(X_test)
lr_prob = logistic_model.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test, lr_pred,
    target_names=["malignant", "benign"]
))


### TODO 4 — Interpret the baseline

Which class has the lower recall? Why could recall be important in a biomedical classification problem?

**Your interpretation:**  
_Type your answer here._


## Step 8 — Compare Three Machine-Learning Models

We will compare:

- Logistic Regression
- Decision Tree
- Random Forest

The goal is **not** simply to find the highest accuracy. We want an evidence-based model choice.


In [ ]:
models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE
    )
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4))


In [ ]:
# Visual comparison
results_df[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].plot(
    kind="bar", figsize=(10,5)
)
plt.ylim(0.70, 1.02)
plt.ylabel("Score")
plt.title("Model Performance Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


### TODO 5 — Model Selection

Do **not** answer only “the model with the highest accuracy.”

Complete:

- **My recommended model:** ...
- **Most important metric for my decision:** ...
- **Evidence from the results:** ...
- **One trade-off:** ...


## Step 9 — Confusion Matrix

A confusion matrix makes the error types visible.


In [ ]:
best_model_name = "Random Forest"  # TODO: change this if your evidence supports another model
best_model = fitted_models[best_model_name]
best_pred = best_model.predict(X_test)

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, best_pred),
    display_labels=["malignant", "benign"]
).plot()

plt.title(f"Confusion Matrix — {best_model_name}")
plt.show()


### TODO 6 — Error Analysis

How many malignant samples were incorrectly predicted as benign?

Why could this error type be important?

**Your answer:**  
_Type your answer here._


## Step 10 — ROC Curve
ROC curves compare discrimination across decision thresholds.


In [ ]:
plt.figure(figsize=(7,6))

for name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=plt.gca())

plt.title("ROC Curves")
plt.show()


### Think & Discuss

Is the model with the best ROC-AUC necessarily the model you would deploy? What other evidence or constraints might matter?


## Step 11 — Feature Importance

Random Forest provides an importance score for each predictor. This helps us ask which variables contribute most to prediction.

> **Important:** predictive importance does **not** establish biological causality.


In [ ]:
rf = fitted_models["Random Forest"]

importance = (
    pd.DataFrame({
        "Feature": X.columns,
        "Importance": rf.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

top10 = importance.head(10)
display(top10)

plt.figure(figsize=(8,5))
plt.barh(top10["Feature"][::-1], top10["Importance"][::-1])
plt.xlabel("Feature Importance")
plt.title("Top 10 Random Forest Features")
plt.tight_layout()
plt.show()


### TODO 7 — Biological Interpretation

List the Top-5 features:

1. ...
2. ...
3. ...
4. ...
5. ...

Then answer:

**Why should we avoid saying that these features “cause” cancer simply because they have high model importance?**

_Type your answer here._


# Step 12 — Model Defense

Prepare a **1–2 minute team defense**.

Complete all five statements:

1. **Our recommended model is:** ...
2. **We selected it because:** ...
3. **The most important evaluation metric is:** ...
4. **The most informative predictive features are:** ...
5. **One important limitation of this experiment is:** ...

Your team should be prepared to defend its choice using evidence rather than accuracy alone.


# Step 13 — Mini Challenge

Choose **ONE** challenge.

### Challenge A — Add a Model
Add SVM, k-NN, or Gradient Boosting and compare it fairly.

### Challenge B — Feature Selection
Train a model using only the Top-10 features and compare it with the full-feature model.

### Challenge C — Cross-Validation
Use stratified 5-fold cross-validation and report **mean F1 ± standard deviation**.


In [ ]:
# Challenge C starter code
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring="f1"
)

print("F1 scores:", np.round(cv_scores, 4))
print(f"Mean F1 ± SD = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


### Challenge Result

- **Challenge selected:** ...
- **What I changed:** ...
- **Result:** ...
- **Did the model improve or become more reliable? Why?** ...


# Step 14 — Final Reflection

Answer briefly:

1. What is the difference between **good training performance** and **good generalization**?
2. What did you learn about using accuracy in biomedical ML?
3. What is one way data leakage could occur?
4. Why is model interpretation important?
5. If this were a real research study, what would you do next?

### Final reflection
_Type your response here._
